# Lab 06: Progressive Disclosure — Skills & Semantic Tool Search

System prompts and tool catalogs both grow over time, and all of it loads on every request, even when the query doesn't touch it. You pay for those tokens every turn, and once they pile up, accuracy suffers too.

Progressive disclosure is the fix: keep a short pointer in context, load the full thing only when a query needs it. It's one tactic in the broader practice of harness engineering, which is about what context, tools, and instructions you hand the model, not just how you word the prompt. We use it on both sides of the agent here:

| | Always in context | Loads on demand | Trigger |
|---|---|---|---|
| **Instructions** — *Skills* | a skill's name + description | the full `SKILL.md` body | the model activates the skill |
| **Tools** — *Semantic search* | (just the search tool) | the matching tool's schema | semantic search per query |

- **Part A — Skills:** pull a long troubleshooting method out of the prompt and into an on-demand skill. The savings are easy to measure on one agent, so we start here.
- **Part B — AgentCore Gateway:** stand up a managed MCP server and fetch only the relevant tool per query with `x_amz_bedrock_agentcore_search`. This is the hands-on build, and most of the lab.

**Prerequisites:** Labs 01–05.

```
01 Baseline → 02 Quick Wins → 03 Caching → 04 Routing → 05 Guardrails → [06 Progressive Disclosure] → 07 Evaluations
                                                                                  ↑ You are here
```

# Part A — Skills: Progressive Disclosure for Instructions

Our system prompt has a big block of troubleshooting instructions baked into it: tone, the questions to ask, one-step-at-a-time pacing, per-symptom decision trees, when to escalate. That's about 1,100 tokens of solid guidance. The problem is it sits in the prompt on every query, and most queries (policy, product, greetings) have nothing to do with a broken device.

A [Strands Agent Skill](https://strandsagents.com/docs/user-guide/concepts/plugins/skills/) (the `AgentSkills` plugin, `strands-agents` 1.30.0 and up) lets us pull that block out of the prompt and package it as something the model loads only when it needs it. A skill is just a folder with a `SKILL.md` file:

- The frontmatter (`name` + `description`) always sits in the prompt, a few dozen tokens.
- The body holds the troubleshooting instructions, and loads only when the model decides to activate the skill by name.

A policy question carries the cheap pointer. A "my laptop won't turn on" pulls in the body. Part B does the same trick for tools; here we're doing it for instructions.

Here's the difference, side by side. Without a skill, all the specialized instructions sit in the prompt the whole time. With one, the model keeps a short description around and only loads the full instructions when a query calls for it.

![Agent with a skill vs. without: without, all instructions stay loaded in context; with, the model loads the full skill only on demand](images/skill%20vs%20without.png)

<sub>Source: [What are Agent Skills, and how agents use them — Daily Dose of Data Science](https://blog.dailydoseofds.com/p/what-are-agent-skills-and-how-agents)</sub>

This pattern isn't Strands-specific. Anthropic published the `SKILL.md` spec as an open standard in December 2025, and within a few months OpenAI Codex, Google Gemini CLI, GitHub Copilot, Cursor, VS Code, JetBrains Junie, and 30-plus other agent tools had adopted it. The skill you write here would work, largely unchanged, across all of them.

### The skill we authored

Run the cell to see the `device-troubleshooting` skill and the prompt sizes side by side. Look closely at the `description`: it lists concrete symptoms and an explicit "don't use it for" list. That's what keeps the skill quiet on unrelated queries, which we'll come back to.

In [1]:
from pathlib import Path

from utils.agent_config import SYSTEM_PROMPT_CORE, SYSTEM_PROMPT_TEXT


# A rough char/4 token estimate is enough to see the relative sizes.
def approx_tokens(text):
    return len(text) // 4


skill_md = Path("agents/skills/device-troubleshooting/SKILL.md").read_text()
print("device-troubleshooting/SKILL.md")
print("=" * 70)
print(skill_md)
print("=" * 70)

print("\nPrompt variants (the troubleshooting method lives in different places):\n")
print(
    f"  v2-v5  SYSTEM_PROMPT_TEXT  (method inlined, paid every query) : ~{approx_tokens(SYSTEM_PROMPT_TEXT):>5} tokens"
)
print(
    f"  v6     SYSTEM_PROMPT_CORE  (method removed, lean prompt)       : ~{approx_tokens(SYSTEM_PROMPT_CORE):>5} tokens"
)
print(f"  v6     SKILL.md body       (loads ONLY when activated)         : ~{approx_tokens(skill_md):>5} tokens")
print(
    f"\n  Prompt shrinks by ~{approx_tokens(SYSTEM_PROMPT_TEXT) - approx_tokens(SYSTEM_PROMPT_CORE)} tokens on EVERY v6 query."
)
print(f"  On a troubleshooting query, the skill adds back only ~{approx_tokens(skill_md)} tokens — still a net win.")

device-troubleshooting/SKILL.md
---
name: device-troubleshooting
description: >-
  Use when a customer reports a malfunctioning device that needs hands-on
  diagnosis — won't power on, won't charge, won't connect, overheats, freezes,
  black screen, or broke after a drop, spill, or update. Do not use for product
  specs, pricing, recommendations, returns, warranties, or greetings.
metadata:
  domain: technical-support
---

# Troubleshooting a Malfunctioning Device

## STEP 0 — REQUIRED before you reply

You MUST call the `get_technical_support` tool first, passing the customer's issue,
and wait for its result. Do NOT write any answer until you have called it — the
knowledge base is the source of truth for the actual fix steps. This skill only
tells you *how* to deliver them; `get_technical_support` supplies *what* to say.
Calling this tool is not optional, even though you already have the guidance below.

Once you have the tool's result, deliver it using the method below.

## How to de

### What this saves, per query

The win is simple: the troubleshooting block only costs tokens when the agent is actually helping someone fix a device. On every other question, it's gone from the prompt.

| What the model carries | Prompt size |
|---|---:|
| Troubleshooting block kept in the prompt | ~2,232 tok |
| Block moved to the skill, on a non-troubleshooting question | ~1,096 tok |
| Block moved to the skill, on a troubleshooting question (skill loads) | ~1,096 + ~312 tok |

So a policy or product question runs on a prompt that's ~1,100 tokens lighter. A troubleshooting question pays ~312 tokens to pull the skill in, still well under the ~1,100 you'd carry on *every* question if the block just lived in the prompt. (These are prompt sizes from the cell above, so they don't move with the model or the answer length.)

You also get a better answer: the skill makes the agent open with "that's frustrating, let's sort it out" and ask one question at a time instead of dumping a checklist.

> Heads up: the per-question input counts in Step 12 will look bigger than these, because they also include tool definitions and the text a tool returns. Those have nothing to do with the skill. The clean number is the prompt delta above.

### The catch: the skill has to stay quiet

The savings only hold if the skill doesn't fire on unrelated queries. Give it a vague description and the model loads it just in case, which adds tokens instead of cutting them. The `description` is what does the routing ([guide](https://agentskills.io/skill-creation/optimizing-descriptions)). What worked was naming the concrete symptoms and adding a "don't use it for product specs, pricing, returns" list.

> Two caveats. Activation isn't deterministic, so tune the description against a trigger rate rather than trusting a single run. And every skill carries a little fixed overhead, so it only pays off when the content you're externalizing is big and the agent reliably skips it. For a tiny snippet you always need, just inline it.

The v6 agent you deploy in Part B already has this wired in. Look for `AgentSkills` and `SYSTEM_PROMPT_CORE` in `agents/v6_skills_and_gateway.py`.

# Part B — Semantic Tool Search: the Same Trick for Tools

Part A loaded instructions on demand. Now we do it for tools, using AgentCore Gateway.

Every tool sends its name, description, and JSON schema into context on every request, before the model has even read the question. At 4 tools that's fine. At 40+ it isn't:

- **Token bloat.** A schema runs ~200–800 tokens. Anthropic measured 58 tools at roughly 55K tokens per turn ([*Advanced tool use*, 2025](https://www.anthropic.com/engineering/advanced-tool-use)).
- **Accuracy falls off.** On the Berkeley Function-Calling Leaderboard, accuracy dropped from 43% to 2% as the tool count went from 4 to 51 ([BFCL](https://gorilla.cs.berkeley.edu/leaderboard)). The model starts hallucinating tools or picking near-duplicates.

Same fix as before: fetch the few relevant tools per query instead of loading the whole catalog. Picking a tool becomes a search problem, not a reasoning one.

### The landscape of tool-search solutions

| Approach | How it picks tools | Token / accuracy effect | Notes |
|---|---|---|---|
| **Static (load all)** | All schemas in context every turn | Baseline; degrades past ~15 tools | Fine for a handful of tools |
| **Embedding retrieval / RAG-MCP** | Embed query + tool descriptions, take top-k | ~50–99% fewer tool tokens; accuracy ~tripled in RAG-MCP | The common first fix ([RAG-MCP, 2025](https://arxiv.org/abs/2505.03275)) |
| **Hybrid + rerank cascade** | Embedding → keyword (BM25) → cross-encoder rerank | Best precision; handles similar tools / exact API terms | For 40+ tools, multi-step workflows |
| **Anthropic Tool Search Tool** | Model calls a built-in search; tools marked `defer_loading` | ~85% fewer tokens; Opus 4 MCP accuracy 49%→74% | Native to the Claude API ([docs](https://www.anthropic.com/engineering/advanced-tool-use)) |
| **AgentCore Gateway** *(this lab)* | Managed MCP server with `searchType: SEMANTIC`; `x_amz_bedrock_agentcore_search` | Same idea, managed on AWS — tools indexed once, shared across agents | What we build below |
| **Code execution / "Code Mode"** | Tools exposed as code APIs; agent loads only what it imports | Anthropic: 150K → 2K tokens (98.7%) on one workflow | Adjacent pattern; also keeps *results* out of context ([Anthropic, 2025](https://www.anthropic.com/engineering/code-execution-with-mcp)) |

In production these often stack: classify the query to a category, embed-retrieve within it, then rerank, which lands you 3–8 tools instead of the whole catalog ([Redis, 2025](https://redis.io/blog/from-reasoning-to-retrieval-solving-the-mcp-tool-overload-problem/)).

What you're saving is the tool-definition tokens, the schemas that load before the query. Fetch 3–5 tools instead of all of them and that overhead stays roughly flat no matter how big the catalog gets, around 48K tokens saved per request at 100 tools.

> One thing to set expectations on: at 4 tools the definition overhead is tiny, so Step 12 won't show a tool-search win, and the input-token swing you do see there is mostly tool results anyway. Treat the Gateway here as the mechanism. The real payoff is at 50–100+ tools. The savings you can actually measure on a small agent are the skill savings from Part A.

## Architecture

![V6 Agent Architecture — AgentCore Gateway with Semantic Tool Search](images/v6_agent_architecture.png)

## Step 1: Setup and Imports

In [2]:
from __future__ import annotations

import base64
import json
import os
import uuid
from pathlib import Path

import httpx
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

import boto3
from bedrock_agentcore_starter_toolkit import Runtime
from mcp.client.streamable_http import streamable_http_client
from strands.tools.mcp import MCPClient

region = os.environ.get("AWS_REGION", os.environ.get("AWS_DEFAULT_REGION", "us-east-1"))
ssm_client = boto3.client("ssm", region_name=region)
gateway_client = boto3.client("bedrock-agentcore-control", region_name=region)
cognito_client = boto3.client("cognito-idp", region_name=region)
agentcore_runtime = Runtime()

print(f"Region: {region}")
print(f"Langfuse Host: {os.environ.get('LANGFUSE_BASE_URL', 'https://cloud.langfuse.com')}")

/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


Region: us-east-1
Langfuse Host: https://d1fnzg75c19u2d.cloudfront.net


---

**What's Different in This Lab?**

In previous labs (v1-v5), tools were **Python functions defined locally** in `utils/tools.py` and loaded directly into the agent:

```python
# v1-v5: Tools loaded directly from local files
from utils.tools import get_return_policy, get_product_info, web_search
agent = Agent(tools=[get_return_policy, get_product_info, ...])
```

In this lab, tools are **registered with AgentCore Gateway** — a centralized MCP server that hosts tools and provides them to agents via the Model Context Protocol (MCP):

```python
# v6: Tools accessed via Gateway
mcp_client = MCPClient(gateway_url)
tools = mcp_client.search_tools(query)  # Semantic search finds relevant tools
agent = Agent(tools=tools)
```

This enables tool sharing across multiple agents and semantic search for relevant tools.

---

## Step 2: Helper Functions

In [3]:
def get_ssm_parameter(name):
    """Retrieve a parameter from SSM Parameter Store."""
    response = ssm_client.get_parameter(Name=name)
    return response["Parameter"]["Value"]


def put_ssm_parameter(name, value):
    """Store a parameter in SSM Parameter Store."""
    ssm_client.put_parameter(Name=name, Value=value, Type="String", Overwrite=True)


def get_cognito_token(client_id, client_secret, token_url, scope):
    """Get OAuth2 token using client_credentials flow."""
    auth = base64.b64encode(f"{client_id}:{client_secret}".encode()).decode()
    response = requests.post(
        token_url,
        headers={
            "Authorization": f"Basic {auth}",
            "Content-Type": "application/x-www-form-urlencoded",
        },
        data={"grant_type": "client_credentials", "scope": scope},
    )
    response.raise_for_status()
    return response.json()["access_token"]


print("Helper functions defined")

Helper functions defined


## Step 3: Get Cognito Configuration

AgentCore Gateway uses JWT authentication. We'll use Cognito to generate tokens.

In [4]:
# Get Cognito configuration from SSM
try:
    cognito_client_id = get_ssm_parameter("/app/customersupport/agentcore/client_id")
    cognito_pool_id = get_ssm_parameter("/app/customersupport/agentcore/pool_id")
    cognito_discovery_url = get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
    cognito_token_url = get_ssm_parameter("/app/customersupport/agentcore/cognito_token_url")
    cognito_scope = get_ssm_parameter("/app/customersupport/agentcore/cognito_auth_scope")

    print("Cognito configuration loaded from SSM:")
    print(f"  Client ID: {cognito_client_id[:20]}...")
    print(f"  Pool ID: {cognito_pool_id}")
    print(f"  Discovery URL: {cognito_discovery_url}")
except Exception as e:
    print(f"Error loading Cognito config: {e}")
    print("Make sure you've deployed the Cognito stack: make deploy-cognito")

Cognito configuration loaded from SSM:
  Client ID: 3pb9lghsc9arkh99k2rd...
  Pool ID: us-east-1_pz5W7bbxC
  Discovery URL: https://cognito-idp.us-east-1.amazonaws.com/us-east-1_pz5W7bbxC/.well-known/openid-configuration


In [5]:
# Get client secret from Cognito
try:
    client_response = cognito_client.describe_user_pool_client(
        UserPoolId=cognito_pool_id,
        ClientId=cognito_client_id,
    )
    client_secret = client_response["UserPoolClient"]["ClientSecret"]
    print("Client secret retrieved successfully")
except Exception as e:
    print(f"Error getting client secret: {e}")
    client_secret = None

Client secret retrieved successfully


In [6]:
# Get bearer token for Gateway authentication
if client_secret:
    bearer_token = get_cognito_token(
        cognito_client_id,
        client_secret,
        cognito_token_url,
        cognito_scope,
    )
    print(f"Bearer token obtained: {bearer_token[:50]}...")
else:
    bearer_token = None
    print("No bearer token - Gateway auth will not work")

Bearer token obtained: eyJraWQiOiJOd3pnL1BOTWFwZjUyQnlmVUdWdmQrY0hLM2hMVV...


---

**Why JWT Authentication?**

Unlike local tools that run in the same process as your agent, Gateway tools are accessed over HTTPS. The Gateway requires authentication to:

- **Control access** — Only authorized agents can use the tools
- **Enable multi-tenancy** — Different teams can have different tool access
- **Audit usage** — Track which agents are calling which tools

The bearer token we just obtained will be passed in the `Authorization` header when connecting to the Gateway.

---

## Step 4: Create AgentCore Gateway

Now we create the Gateway itself — a managed MCP server that will host our tools.

In [7]:
import time


def wait_for_gateway_ready(gateway_id, timeout_seconds=120, poll_interval=5):
    """Poll gateway status until READY or timeout."""
    start_time = time.time()
    while time.time() - start_time < timeout_seconds:
        response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
        status = response.get("status")
        if status == "READY":
            return True
        if status == "FAILED":
            raise Exception(f"Gateway creation failed: {response.get('statusReasons', 'Unknown reason')}")
        print(f"  Gateway status: {status}, waiting...")
        time.sleep(poll_interval)
    raise Exception(f"Timeout waiting for gateway to become READY (current status: {status})")


gateway_name = "workshop-customer-support-gateway"

# Get Gateway IAM role from SSM
gateway_role_arn = get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role")
print(f"Gateway IAM Role: {gateway_role_arn}")

# Configure JWT authorizer
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [cognito_client_id],
        "discoveryUrl": cognito_discovery_url,
    }
}

# Enable semantic search - this creates a vector store for tool discovery
search_config = {"mcp": {"searchType": "SEMANTIC", "supportedVersions": ["2025-03-26"]}}

try:
    # Create new gateway with semantic search enabled
    print(f"Creating gateway: {gateway_name}")
    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn=gateway_role_arn,
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration=auth_config,
        protocolConfiguration=search_config,  # Enable semantic search
        description="Customer Support Workshop Gateway with Semantic Search",
    )

    gateway_id = create_response["gatewayId"]
    gateway_url = create_response["gatewayUrl"]
    gateway_arn = create_response["gatewayArn"]

    # Store in SSM for later use
    put_ssm_parameter("/app/customersupport/agentcore/gateway_id", gateway_id)
    put_ssm_parameter("/app/customersupport/agentcore/gateway_url", gateway_url)
    put_ssm_parameter("/app/customersupport/agentcore/gateway_arn", gateway_arn)

    # Wait for gateway to be ready before proceeding
    print("Waiting for gateway to become ready...")
    wait_for_gateway_ready(gateway_id)

    print("Gateway created successfully with semantic search!")
    print(f"  ID: {gateway_id}")
    print(f"  URL: {gateway_url}")

except gateway_client.exceptions.ConflictException:
    # Gateway already exists, get existing (no wait needed - already ready)
    print(f"Gateway '{gateway_name}' already exists, retrieving...")
    gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
    gateway_url = gateway_response["gatewayUrl"]
    gateway_arn = gateway_response["gatewayArn"]
    print("Using existing gateway:")
    print(f"  ID: {gateway_id}")
    print(f"  URL: {gateway_url}")

except Exception as e:
    print(f"Error creating gateway: {e}")
    gateway_id = None
    gateway_url = None

Gateway IAM Role: arn:aws:iam::739907928487:role/prompt-workshop-infra-GatewayAgentCoreRole-CjWmvtPukRln
Creating gateway: workshop-customer-support-gateway
Gateway 'workshop-customer-support-gateway' already exists, retrieving...
Using existing gateway:
  ID: workshop-customer-support-gateway-rgkzl6gn83
  URL: https://workshop-customer-support-gateway-rgkzl6gn83.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp


---

**What is AgentCore Gateway?**

AgentCore Gateway is a **managed MCP (Model Context Protocol) server** that:

1. **Hosts tools centrally** — Tools are registered once and shared across agents
2. **Provides semantic search** — The `protocolConfiguration` with `searchType: "SEMANTIC"` enables vector-based tool discovery
3. **Handles authentication** — JWT tokens control who can access which tools
4. **Exposes an MCP endpoint** — Agents connect via the Gateway URL using standard MCP protocol

The Gateway URL (`/mcp` endpoint) is what our agent will connect to instead of loading tools locally.

---

## Step 5: Deploy Lambda for Gateway Tools

The Gateway routes tool calls to a **Lambda function** (the "target"). In previous labs, tools were Python functions running inside the agent process. Now we deploy them as a standalone Lambda that the Gateway invokes.

We'll create:
1. An **IAM role** for the Lambda with permissions for Bedrock Knowledge Base and SSM
2. A **Lambda function** implementing all 4 customer support tools
3. Attach the existing **DDGS layer** for web search capability

In [8]:
# Lambda function code implementing all 4 customer support tools
# These are the same tools from utils/tools.py, adapted for Lambda execution

lambda_code = '''
import json
import boto3
from ddgs import DDGS


def get_return_policy(product_category):
    """Get return policy information for a specific product category."""
    return_policies = {
        "smartphones": {
            "window": "30 days",
            "condition": "Original packaging, no physical damage, factory reset required",
            "process": "Online RMA portal or technical support",
            "refund_time": "5-7 business days after inspection",
            "shipping": "Free return shipping, prepaid label provided",
            "warranty": "1-year manufacturer warranty included",
            "restocking_fee": "None for unopened, 15% for opened items",
        },
        "laptops": {
            "window": "30 days",
            "condition": "Original packaging, all accessories included, no physical damage",
            "process": "Online RMA portal or in-store return",
            "refund_time": "7-10 business days after inspection",
            "shipping": "Free return shipping for defects, customer pays for change of mind",
            "warranty": "1-year manufacturer warranty, extended warranty available",
            "restocking_fee": "None for defects, 15% for change of mind",
        },
        "accessories": {
            "window": "60 days",
            "condition": "Unopened original packaging preferred, opened items accepted",
            "process": "Online RMA portal, mail-in, or in-store",
            "refund_time": "3-5 business days after inspection",
            "shipping": "Free return shipping on orders over $50",
            "warranty": "90-day manufacturer warranty",
            "restocking_fee": "None",
        },
        "tablets": {
            "window": "30 days",
            "condition": "Original packaging, factory reset, no physical damage",
            "process": "Online RMA portal or technical support",
            "refund_time": "5-7 business days after inspection",
            "shipping": "Free return shipping, prepaid label provided",
            "warranty": "1-year manufacturer warranty included",
            "restocking_fee": "None for unopened, 10% for opened items",
        },
        "audio": {
            "window": "45 days",
            "condition": "Original packaging, all accessories included",
            "process": "Online RMA portal or in-store",
            "refund_time": "5-7 business days after inspection",
            "shipping": "Free return shipping",
            "warranty": "1-year manufacturer warranty",
            "restocking_fee": "None",
        },
    }

    default_policy = {
        "window": "30 days",
        "condition": "Original packaging, undamaged condition",
        "process": "Contact customer support for RMA",
        "refund_time": "7-10 business days after inspection",
        "shipping": "Varies by product",
        "warranty": "Standard manufacturer warranty",
        "restocking_fee": "May apply",
    }

    policy = return_policies.get(product_category.lower(), default_policy)
    category_name = product_category.title()

    return f"""Return Policy - {category_name}:

- Return Window: {policy["window"]}
- Condition Requirements: {policy["condition"]}
- Return Process: {policy["process"]}
- Refund Timeline: {policy["refund_time"]}
- Shipping: {policy["shipping"]}
- Warranty: {policy["warranty"]}
- Restocking Fee: {policy["restocking_fee"]}

For assistance with returns, visit our online RMA portal or contact customer support."""


def get_product_info(product_type):
    """Get detailed technical specifications for electronics products."""
    products = {
        "laptops": {
            "featured_model": "TechMart Pro 15",
            "price": "$1,299",
            "specs": {
                "processor": "Intel Core i7-13700H (14 cores, up to 5.0GHz)",
                "memory": "16GB DDR5 RAM (upgradeable to 32GB)",
                "storage": "512GB NVMe SSD (expandable)",
                "display": "15.6 FHD IPS, 144Hz refresh rate, 300 nits",
                "graphics": "NVIDIA RTX 4060 6GB",
                "battery": "72Wh, up to 10 hours",
                "weight": "4.5 lbs (2.04 kg)",
                "ports": "2x USB-C, 2x USB-A, HDMI 2.1, SD card reader",
            },
            "colors": ["Space Gray", "Silver", "Midnight Blue"],
            "in_stock": True,
            "delivery": "2-3 business days",
        },
        "smartphones": {
            "featured_model": "TechMart Phone X",
            "price": "$899",
            "specs": {
                "processor": "Snapdragon 8 Gen 2",
                "memory": "8GB RAM",
                "storage": "256GB (no expansion)",
                "display": "6.7 AMOLED, 120Hz, 2400x1080",
                "camera": "50MP main + 12MP ultrawide + 10MP telephoto",
                "battery": "5000mAh, 65W fast charging",
                "os": "Android 14",
                "connectivity": "5G, Wi-Fi 6E, Bluetooth 5.3",
            },
            "colors": ["Obsidian Black", "Pearl White", "Ocean Blue"],
            "in_stock": True,
            "delivery": "1-2 business days",
        },
        "tablets": {
            "featured_model": "TechMart Tab Pro",
            "price": "$649",
            "specs": {
                "processor": "Apple M2 chip",
                "memory": "8GB RAM",
                "storage": "128GB / 256GB / 512GB options",
                "display": "11 Liquid Retina, 120Hz ProMotion",
                "camera": "12MP rear + 12MP front with Center Stage",
                "battery": "Up to 10 hours",
                "connectivity": "Wi-Fi 6E, optional 5G",
                "accessories": "Compatible with Magic Keyboard and Apple Pencil",
            },
            "colors": ["Silver", "Space Gray"],
            "in_stock": True,
            "delivery": "2-3 business days",
        },
        "audio": {
            "featured_model": "TechMart SoundPro Headphones",
            "price": "$349",
            "specs": {
                "type": "Over-ear wireless",
                "driver": "40mm custom drivers",
                "anc": "Active Noise Cancellation with transparency mode",
                "battery": "30 hours (ANC on), 40 hours (ANC off)",
                "connectivity": "Bluetooth 5.2, 3.5mm jack",
                "features": "Multipoint connection, spatial audio",
                "weight": "250g",
                "charging": "USB-C, 10min charge = 3 hours playback",
            },
            "colors": ["Midnight Black", "Cloud White", "Forest Green"],
            "in_stock": True,
            "delivery": "1-2 business days",
        },
        "accessories": {
            "featured_items": [
                {"name": "USB-C Hub 7-in-1", "price": "$49", "in_stock": True},
                {"name": "Wireless Charging Pad", "price": "$29", "in_stock": True},
                {"name": "Laptop Stand Adjustable", "price": "$39", "in_stock": True},
                {"name": "Screen Protector Kit", "price": "$19", "in_stock": True},
                {"name": "Premium Laptop Bag", "price": "$79", "in_stock": True},
            ],
            "delivery": "1-2 business days",
        },
    }

    product = products.get(product_type.lower())

    if not product:
        return f"Product type '{product_type}' not found. Available categories: {', '.join(products.keys())}"

    if product_type.lower() == "accessories":
        items_list = "\\n".join([f"  - {item['name']}: {item['price']}" for item in product["featured_items"]])
        return f"""Accessories Collection:

Featured Items:
{items_list}

Delivery: {product["delivery"]}

All accessories come with a 90-day warranty."""

    specs_list = "\\n".join([f"  - {k.title()}: {v}" for k, v in product["specs"].items()])
    colors_list = ", ".join(product.get("colors", []))

    return f"""{product["featured_model"]} - {product["price"]}

Specifications:
{specs_list}

Available Colors: {colors_list}
In Stock: {"Yes" if product["in_stock"] else "No"}
Delivery: {product["delivery"]}

Contact sales for bulk orders or custom configurations."""


def get_technical_support(issue_description):
    """Query Bedrock Knowledge Base for troubleshooting and technical assistance."""
    try:
        ssm = boto3.client("ssm")
        sts = boto3.client("sts")
        account_id = sts.get_caller_identity()["Account"]
        region_name = boto3.Session().region_name
        kb_id = ssm.get_parameter(
            Name=f"/{account_id}-{region_name}/kb/knowledge-base-id"
        )["Parameter"]["Value"]

        bedrock_agent = boto3.client("bedrock-agent-runtime")
        response = bedrock_agent.retrieve(
            knowledgeBaseId=kb_id,
            retrievalQuery={"text": issue_description},
            retrievalConfiguration={
                "vectorSearchConfiguration": {"numberOfResults": 3}
            },
        )

        results = response.get("retrievalResults", [])
        if not results:
            return "No relevant technical support documentation found."

        formatted = []
        for r in results:
            text = r.get("content", {}).get("text", "")
            score = r.get("score", 0)
            if score >= 0.4 and text:
                formatted.append(text)

        if formatted:
            return "\\n\\n---\\n\\n".join(formatted)
        return "No relevant technical support results found above confidence threshold."

    except Exception as e:
        return f"Unable to access technical support documentation. Error: {str(e)}"


def web_search(keywords, max_results=5):
    """Search the web for updated information using DuckDuckGo."""
    try:
        results = DDGS().text(keywords, region="us-en", max_results=int(max_results))
        if not results:
            return "No search results found for the given keywords."

        formatted = []
        for i, r in enumerate(results, 1):
            formatted.append(
                f"{i}. {r.get('title', 'No title')}\\n"
                f"   URL: {r.get('href', 'No URL')}\\n"
                f"   {r.get('body', 'No description')}"
            )
        return "Web Search Results:\\n\\n" + "\\n\\n".join(formatted)

    except Exception as e:
        return f"Search error: {str(e)}"


def lambda_handler(event, context):
    """Route tool calls to the appropriate handler."""
    print(f"Event: {event}")

    extended_tool_name = context.client_context.custom["bedrockAgentCoreToolName"]
    tool_name = extended_tool_name.split("___")[1]
    print(f"Tool: {tool_name}")

    if tool_name == "get_return_policy":
        product_category = event.get("product_category", "")
        if not product_category:
            return {"statusCode": 400, "body": "Please provide product_category"}
        return {"statusCode": 200, "body": get_return_policy(product_category)}

    elif tool_name == "get_product_info":
        product_type = event.get("product_type", "")
        if not product_type:
            return {"statusCode": 400, "body": "Please provide product_type"}
        return {"statusCode": 200, "body": get_product_info(product_type)}

    elif tool_name == "get_technical_support":
        issue_description = event.get("issue_description", "")
        if not issue_description:
            return {"statusCode": 400, "body": "Please provide issue_description"}
        return {"statusCode": 200, "body": get_technical_support(issue_description)}

    elif tool_name == "web_search":
        keywords = event.get("keywords", "")
        if not keywords:
            return {"statusCode": 400, "body": "Please provide keywords for search"}
        max_results = event.get("max_results", 5)
        return {"statusCode": 200, "body": web_search(keywords, max_results)}

    return {"statusCode": 400, "body": f"Unknown tool: {tool_name}"}
'''

print(f"Lambda code defined ({len(lambda_code)} characters)")

Lambda code defined (11801 characters)


In [9]:
import io
import zipfile

lambda_client = boto3.client("lambda", region_name=region)
iam_client = boto3.client("iam")
account_id = boto3.client("sts").get_caller_identity()["Account"]

# --- 1. Reuse IAM role from the CloudFormation-deployed Lambda ---
# The prerequisite stack already created a Lambda role with the right permissions
# (SSM, DynamoDB, CloudWatch, Bedrock). We reuse it instead of creating a new one.
existing_lambda_arn = get_ssm_parameter("/app/customersupport/agentcore/lambda_arn")
existing_lambda_config = lambda_client.get_function_configuration(FunctionName=existing_lambda_arn)
lambda_role_arn = existing_lambda_config["Role"]
print(f"Reusing existing Lambda role: {lambda_role_arn}")

# --- 2. Find DDGS layer from prerequisite stack ---
layers_response = lambda_client.list_layers()
ddgs_layer_arn = None
for layer in layers_response["Layers"]:
    if "ddgs" in layer["LayerName"].lower():
        ddgs_layer_arn = layer["LatestMatchingVersion"]["LayerVersionArn"]
        break

if ddgs_layer_arn:
    print(f"Found DDGS layer: {ddgs_layer_arn}")
else:
    print("WARNING: DDGS layer not found — web_search tool will not work")

# --- 3. Package and deploy Lambda ---
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", lambda_code)
zip_bytes = zip_buffer.getvalue()

gateway_lambda_name = "workshop-gateway-tools"
lambda_layers = [ddgs_layer_arn] if ddgs_layer_arn else []

try:
    create_response = lambda_client.create_function(
        FunctionName=gateway_lambda_name,
        Runtime="python3.12",
        Role=lambda_role_arn,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_bytes},
        Layers=lambda_layers,
        Timeout=30,
        MemorySize=256,
        Environment={"Variables": {"PYTHONPATH": "/opt/python"}},
        Description="Gateway tools for customer support workshop",
    )
    lambda_arn = create_response["FunctionArn"]
    print(f"Lambda created: {lambda_arn}")

except lambda_client.exceptions.ResourceConflictException:
    # Update existing function
    lambda_client.update_function_code(
        FunctionName=gateway_lambda_name,
        ZipFile=zip_bytes,
    )
    lambda_arn = f"arn:aws:lambda:{region}:{account_id}:function:{gateway_lambda_name}"
    print(f"Lambda updated: {lambda_arn}")

# --- 4. Grant Gateway role permission to invoke this Lambda ---
gateway_role_arn = get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role")
gateway_role_name = gateway_role_arn.split("/")[-1]

try:
    iam_client.put_role_policy(
        RoleName=gateway_role_name,
        PolicyName="InvokeGatewayLambdaPolicy",
        PolicyDocument=json.dumps(
            {
                "Version": "2012-10-17",
                "Statement": [
                    {
                        "Effect": "Allow",
                        "Action": "lambda:InvokeFunction",
                        "Resource": lambda_arn,
                    }
                ],
            }
        ),
    )
    print(f"Granted Gateway role permission to invoke Lambda")
    # IAM policy propagation takes time — wait before creating Gateway target
    print("Waiting 15s for IAM policy propagation...")
    time.sleep(15)
except Exception as e:
    print(f"ERROR: Could not update Gateway role: {e}")
    print(f"The Gateway role must have lambda:InvokeFunction on {lambda_arn}")

print(f"\nLambda ARN for Gateway target: {lambda_arn}")

Reusing existing Lambda role: arn:aws:iam::739907928487:role/service-role/prompt-workshop-infra-CustomerSupportLambdaRole-SRXYkAjwuzUh
Found DDGS layer: arn:aws:lambda:us-east-1:739907928487:layer:prompt-workshop-infra-ddgs-layer:2
Lambda updated: arn:aws:lambda:us-east-1:739907928487:function:workshop-gateway-tools
Granted Gateway role permission to invoke Lambda
Waiting 15s for IAM policy propagation...

Lambda ARN for Gateway target: arn:aws:lambda:us-east-1:739907928487:function:workshop-gateway-tools


## Step 6: Register Tools with Gateway

Now we register our tools with the Gateway. Instead of Python functions, we define **tool schemas** that describe each tool's name, description, and parameters.

In [10]:
# Tool schemas for all customer support tools
# These descriptions are indexed for semantic search
tool_schema = [
    {
        "name": "get_return_policy",
        "description": "Get return policy information for a product category including return windows, conditions, refund timelines, and warranty coverage. Use for questions about returns, refunds, exchanges, or warranties.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "product_category": {
                    "type": "string",
                    "description": "Product category (smartphones, laptops, tablets, audio, accessories)",
                }
            },
            "required": ["product_category"],
        },
    },
    {
        "name": "get_product_info",
        "description": "Get detailed product specifications, pricing, and availability for electronics. Use for questions about product features, specs, prices, or comparisons.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "product_type": {
                    "type": "string",
                    "description": "Type of product (laptops, smartphones, tablets, audio, accessories)",
                }
            },
            "required": ["product_type"],
        },
    },
    {
        "name": "get_technical_support",
        "description": "Get troubleshooting help and technical support from the knowledge base. Use for technical issues, setup problems, device malfunctions, or maintenance questions.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "issue_description": {"type": "string", "description": "Description of the technical issue or problem"}
            },
            "required": ["issue_description"],
        },
    },
    {
        "name": "web_search",
        "description": "Search the web for current information, recent news, promotions, or updates. Use when the user asks about current events, latest news, or time-sensitive information.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "keywords": {"type": "string", "description": "Search query keywords"},
                "max_results": {"type": "integer", "description": "Maximum number of results to return (default: 5)"},
            },
            "required": ["keywords"],
        },
    },
]

print(f"Defined {len(tool_schema)} tool schemas for Gateway:")
for tool in tool_schema:
    print(f"  - {tool['name']}")

Defined 4 tool schemas for Gateway:
  - get_return_policy
  - get_product_info
  - get_technical_support
  - web_search


---

**From Python Functions to Tool Schemas**

In previous labs, tools were Python functions with docstrings:

```python
# v1-v5: Python function
def get_return_policy(product_category: str) -> str:
    """Get return policy for a product category."""
    ...
```

For Gateway registration, we define the **same tools as JSON schemas**. The schema includes:
- `name` — Tool identifier
- `description` — Used for semantic search indexing (be descriptive!)
- `inputSchema` — JSON Schema defining parameters

The actual tool implementation lives in the **Lambda function** we deployed in Step 5 — the Gateway routes tool calls to this Lambda.

---

In [11]:
if gateway_id:
    # Use the Lambda ARN from Step 5 deployment
    print(f"Lambda ARN: {lambda_arn}")

    # Configure Lambda target with all tools
    lambda_target_config = {
        "mcp": {
            "lambda": {
                "lambdaArn": lambda_arn,
                "toolSchema": {"inlinePayload": tool_schema},
            }
        }
    }

    credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]
    target_name = "customer-support-tools"

    # Helper: delete existing target and wait for it to be fully removed
    def delete_existing_target():
        response = gateway_client.list_gateway_targets(gatewayIdentifier=gateway_id)
        # API returns targets under "items" key
        targets_list = response.get("items", response.get("targets", []))
        for target in targets_list:
            if target_name in target.get("name", ""):
                target_id = target["targetId"]
                print(f"  Deleting existing target: {target_id}")
                gateway_client.delete_gateway_target(
                    gatewayIdentifier=gateway_id,
                    targetId=target_id,
                )
                # Poll until target is fully deleted
                for attempt in range(24):  # up to 2 minutes
                    try:
                        gateway_client.get_gateway_target(
                            gatewayIdentifier=gateway_id,
                            targetId=target_id,
                        )
                        print(f"  Waiting for target deletion... ({(attempt + 1) * 5}s)")
                        time.sleep(5)
                    except Exception:
                        print(f"  Target deleted successfully")
                        return True
                print("  WARNING: Target deletion timed out")
                return False
        print("  No matching target found in list — it may have a different name")
        return True

    try:
        # Try creating the target
        target_response = gateway_client.create_gateway_target(
            gatewayIdentifier=gateway_id,
            name=target_name,
            description="Customer support tools with semantic search",
            targetConfiguration=lambda_target_config,
            credentialProviderConfigurations=credential_config,
        )
        print(f"Gateway target created: {target_response['targetId']}")
        print(f"Registered {len(tool_schema)} tools for semantic search")

    except gateway_client.exceptions.ConflictException:
        # Target already exists — delete and recreate with new Lambda ARN
        print(f"Gateway target '{target_name}' already exists. Replacing...")
        delete_existing_target()

        # Recreate with new Lambda ARN
        target_response = gateway_client.create_gateway_target(
            gatewayIdentifier=gateway_id,
            name=target_name,
            description="Customer support tools with semantic search",
            targetConfiguration=lambda_target_config,
            credentialProviderConfigurations=credential_config,
        )
        print(f"Gateway target recreated: {target_response['targetId']}")
        print(f"Registered {len(tool_schema)} tools for semantic search")

    except Exception as e:
        print(f"Error creating gateway target: {e}")
else:
    print("Skipping target creation - no gateway")

Lambda ARN: arn:aws:lambda:us-east-1:739907928487:function:workshop-gateway-tools
Gateway target 'customer-support-tools' already exists. Replacing...
  Deleting existing target: LDVOY7O7CG
  Waiting for target deletion... (5s)
  Target deleted successfully
Gateway target recreated: 5WHXEAE9EL
Registered 4 tools for semantic search


---

**What is a Gateway Target?**

A **Gateway Target** connects tool schemas to their implementations:

```
Agent → Gateway → Target (Lambda) → Tool execution
```

- **Gateway** — Routes requests and handles auth
- **Target** — The backend that executes tools (Lambda, in our case)
- **Tool Schema** — Tells the Gateway what tools exist and their parameters

When an agent calls a tool via MCP, the Gateway:
1. Validates the request against the schema
2. Forwards it to the Lambda target
3. Returns the result to the agent

This decouples tool definitions from implementations — you can update Lambda code without changing schemas.

---

## Step 7: Connect to Gateway via MCP

The agent connects to the Gateway using the MCP (Model Context Protocol) client. When a query comes in, the agent calls `x_amz_bedrock_agentcore_search` to find relevant tools based on the query semantics.

---

**What is MCP (Model Context Protocol)?**

MCP is an open protocol for connecting AI models to external tools and data. Think of it as a standardized way for agents to discover and call tools.

The `MCPClient` connects to the Gateway's MCP endpoint and can:
- **List available tools** — `list_tools()`
- **Call tools** — `call_tool(name, arguments)`
- **Search tools** — `x_amz_bedrock_agentcore_search` (Gateway's semantic search)

By using MCP, our agent can work with any MCP-compatible tool server, not just this specific Gateway.

---

In [12]:
if gateway_url and bearer_token:
    print(f"Gateway Endpoint: {gateway_url}")

    # Set up MCP client with authentication
    mcp_client = MCPClient(
        lambda: streamable_http_client(
            gateway_url,
            http_client=httpx.AsyncClient(
                headers={"Authorization": f"Bearer {bearer_token}"},
            ),
        )
    )
    print("MCP client configured")
else:
    mcp_client = None
    print("MCP client not available - missing gateway URL or token")

Gateway Endpoint: https://workshop-customer-support-gateway-rgkzl6gn83.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp
MCP client configured


## Step 8: Test Semantic Search Locally

Before deploying, let's test the semantic search tool directly via MCP.

In [13]:
# Wait for Gateway vector store to index tools
print("Waiting for Gateway to index tools...")
time.sleep(10)

# Test semantic search directly
if mcp_client:
    with mcp_client:
        # Search for tools related to return policy
        search_result = mcp_client.call_tool_sync(
            tool_use_id=str(uuid.uuid4()),
            name="x_amz_bedrock_agentcore_search",
            arguments={"query": "return policy for laptops"},
        )

        # Extract tools from structuredContent
        tools_found = search_result["structuredContent"]["tools"]
        print("Search query: 'return policy for laptops'")
        print(f"Tools found: {len(tools_found)}")
        for tool in tools_found:
            print(f"  - {tool['name']}")
else:
    print("MCP client not available")

Waiting for Gateway to index tools...
Search query: 'return policy for laptops'
Tools found: 4
  - customer-support-tools___get_return_policy
  - customer-support-tools___get_product_info
  - customer-support-tools___get_technical_support
  - customer-support-tools___web_search


---

**How Semantic Search Works**

The `x_amz_bedrock_agentcore_search` tool is a **built-in Gateway capability** that:

1. Takes a natural language query
2. Converts it to a vector embedding
3. Searches the indexed tool descriptions
4. Returns tools ranked by semantic relevance

The first tool in the results is the **most relevant** to the query. With only 4 tools, the Gateway returns all of them — but notice the **ranking order** reflects semantic similarity to the query.

---

## Step 9: Deploy the v6 Agent

Now let's deploy the agent that uses semantic search to AgentCore Runtime.

In [14]:
# Review the v6 agent code
agent_file = Path("agents/v6_skills_and_gateway.py")
print(agent_file.read_text())

"""
V6 agent - focused on Agent Skills and the MCP Gateway. Progressive disclosure on
both sides: an on-demand troubleshooting skill (instructions) plus AgentCore
Gateway with semantic tool search (tools).
"""

from __future__ import annotations

import base64
import os
import uuid
from pathlib import Path

import httpx
import requests
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from mcp.client.streamable_http import streamable_http_client
from mcp.types import Tool as MCPTool
from strands import Agent
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from strands.tools.mcp import MCPClient
from strands.tools.mcp.mcp_client import MCPAgentTool
from strands.types.content import SystemContentBlock
from strands.vended_plugins.skills import AgentSkills

from utils.agent_config import (
    MODEL_HAIKU,
    MODEL_SONNET,
    SYSTEM_PROMPT_CORE,
    classify_query_complexity,
    setup_langfuse_telemetry,
)

setup_langfuse_telemetry()

app =

---

**Key Differences from v4/v5**

Notice what's **no longer** in the agent code:

```python
# v4/v5 had local tool imports:
from utils.tools import get_return_policy, get_product_info, web_search
```

Instead, the v6 agent:

1. **Connects to Gateway** via `MCPClient` with JWT authentication
2. **Searches for relevant tools** using `x_amz_bedrock_agentcore_search`
3. **Converts search results** to `MCPAgentTool` objects the agent can use
4. **Creates agent with dynamic tools** — different queries may load different tools

The tools themselves still exist (in Lambda), but the agent discovers them at runtime via the Gateway rather than importing them at startup.

---

## Step 10: Configure and Deploy

The v6 agent uses `x_amz_bedrock_agentcore_search` to find relevant tools per query, reducing context size.

In [15]:
agent_name = "customer_support_v6_gateway"
agent_file = str(Path("agents/v6_skills_and_gateway.py").absolute())
requirements_file = str(Path("requirements-for-agentcore.txt").absolute())

# Clean up any existing build files from previous labs
for f in ["Dockerfile", ".dockerignore", ".bedrock_agentcore.yaml"]:
    p = Path(f)
    if p.exists():
        p.unlink()
        print(f"Removed existing: {f}")

print(f"Configuring agent: {agent_name}")
agentcore_runtime.configure(
    entrypoint=agent_file,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file=requirements_file,
    region=region,
    agent_name=agent_name,
)

Entrypoint parsed: file=/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/agents/v6_skills_and_gateway.py, bedrock_agentcore_name=v6_skills_and_gateway
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: customer_support_v6_gateway


Removed existing: Dockerfile
Removed existing: .dockerignore
Removed existing: .bedrock_agentcore.yaml
Configuring agent: customer_support_v6_gateway


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Generated Dockerfile: 
/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/Dockerfile

Generated .dockerignore: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/.dockerignore
Setting 'customer_support_v6_gateway' as default agent
Bedrock AgentCore configured: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/Dockerfile'), dockerignore_path=PosixPath('/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/03-developer-journey/.dockerignore'), runtime='None', runtime_type=None, region='us-east-1', account_id='739907928487', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

In [16]:
# Modify Dockerfile for Langfuse
import re

dockerfile_path = Path("Dockerfile")
if dockerfile_path.exists():
    content = dockerfile_path.read_text()
    if "opentelemetry-instrument" in content:
        content = re.sub(
            r'CMD \["opentelemetry-instrument", "python", "-m", "([^"]+)"\]', r'CMD ["python", "-m", "\1"]', content
        )
        dockerfile_path.write_text(content)
        print("Dockerfile modified for Langfuse")
    else:
        print("Dockerfile already configured")

Dockerfile modified for Langfuse


In [17]:
env_vars = {
    "LANGFUSE_BASE_URL": os.environ.get("LANGFUSE_BASE_URL"),
    "LANGFUSE_PUBLIC_KEY": os.environ.get("LANGFUSE_PUBLIC_KEY"),
    "LANGFUSE_SECRET_KEY": os.environ.get("LANGFUSE_SECRET_KEY"),
    "GATEWAY_URL": gateway_url or "",
    "GUARDRAIL_ID": os.environ.get("GUARDRAIL_ID", ""),
    # Cognito credentials for Gateway authentication
    "COGNITO_CLIENT_ID": cognito_client_id,
    "COGNITO_CLIENT_SECRET": client_secret,
    "COGNITO_TOKEN_URL": cognito_token_url,
    "COGNITO_SCOPE": cognito_scope,
    "PYTHONUNBUFFERED": "1",
}

print("Deploying to AgentCore Runtime...")
print(f"  Gateway URL: {gateway_url}")
print(f"  Cognito credentials: {'configured' if client_secret else 'missing'}")
launch_result = agentcore_runtime.launch(env_vars=env_vars, auto_update_on_conflict=True)
agent_arn = launch_result.agent_arn
print(f"Agent deployed: {agent_arn}")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'customer_support_v6_gateway' to account 739907928487 (us-east-1)
Generated image tag: 20260602-173559-720
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: customer_support_v6_gateway


Deploying to AgentCore Runtime...
  Gateway URL: https://workshop-customer-support-gateway-rgkzl6gn83.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp
  Cognito credentials: configured


ECR repository available: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_v6_gateway
Getting or creating execution role for agent: customer_support_v6_gateway
Using AWS region: us-east-1, account ID: 739907928487
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-74b3989be9


✅ Reusing existing ECR repository: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-customer_support_v6_gateway


✅ Reusing existing execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-74b3989be9
Execution role available: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-74b3989be9
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: customer_support_v6_gateway
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-74b3989be9
Reusing existing CodeBuild execution role: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-74b3989be9
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: customer_support_v6_gateway/source.zip
Updated CodeBuild project: bedrock-agentcore-customer_support_v6_gateway-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.6s
🔄 PROVISIONING started (total: 2s)
✅ PROVISIONING completed in 7.6s
🔄 DOWNLOAD_SO

Agent deployed: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/customer_support_v6_gateway-gtZd6HBB4d


In [18]:
# Save the agent ARN for later use
agent_arn = launch_result.agent_arn
print(f"Agent ARN: {agent_arn}")

# Grant the auto-created execution role the shared runtime permissions.
# v6 tools run in the Gateway Lambda (not the agent), so the KB statements are no-ops
# here — but the agent still applies the guardrail, which needs bedrock:ApplyGuardrail.
from utils.runtime_helpers import ensure_runtime_permissions

ensure_runtime_permissions(region)

Agent ARN: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/customer_support_v6_gateway-gtZd6HBB4d
Granted customer-support runtime permissions to AmazonBedrockAgentCoreSDKRuntime-us-east-1-74b3989be9


## Step 11: Define Invocation Helper

In [19]:
data_client = boto3.client("bedrock-agentcore", region_name=region)


def invoke_agent(prompt):
    """Invoke the deployed agent via AgentCore API."""
    response = data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    return json.loads(response["response"].read().decode("utf-8"))

In [20]:
# Import Langfuse metrics helper
from utils.langfuse_metrics import (
    clear_metrics,
    collect_metric,
    get_latest_trace_metrics,
    print_metrics,
    print_metrics_table,
)

# Clear any previously collected metrics
clear_metrics()
print("Metrics helper ready")

Metrics helper ready


## Step 12: Test the Agent

Run the same test prompts as previous labs to compare results.

> **Note:** For this demo, we take only the **top-ranked tool** from semantic search results. With only 4 tools, the Gateway returns all of them as potentially relevant. In production with 100+ tools, the search would naturally filter to a smaller relevant subset.

In [21]:
# Same test prompts as v2, v3, v4 notebooks for consistent comparison
TEST_PROMPTS = [
    ("Return Policy", "What is your return policy for laptops?"),
    ("Product Info", "Tell me about your smartphone options"),
    ("Technical Support", "My laptop won't turn on, can you help me troubleshoot?"),
    ("Multi-part Question", "I want to buy a laptop. What are the specs and what's the return policy?"),
    ("General Question", "Hello! What can you help me with today?"),
]

# Run all tests
print("=" * 70)
print("V6 GATEWAY WITH SEMANTIC TOOL SEARCH")
print("=" * 70)
print("Each query uses x_amz_bedrock_agentcore_search to find relevant tools.")
print("Watch two things:")
print("  - tools_loaded     : semantic search picks the relevant tool(s)")
print("  - skill_activated  : the device-troubleshooting skill should fire ONLY")
print("                       on the malfunction query, and stay dormant otherwise")
print("=" * 70)

for test_name, prompt in TEST_PROMPTS:
    print("\n" + "-" * 70)
    print(f"Test: {test_name}")
    print(f"Query: {prompt}")
    print("-" * 70)

    response = invoke_agent(prompt)

    # Show tools_loaded (semantic search) and skill_activated (progressive-disclosure
    # of instructions). skill_activated should be True only for Technical Support.
    tools_loaded = response.get("tools_loaded", "N/A")
    skill_activated = response.get("skill_activated", "N/A")
    print(f"Tools Loaded:     {tools_loaded} (via semantic search)")
    print(f"Skill Activated:  {skill_activated} (device-troubleshooting)")
    print(f"Response: {str(response.get('response', response))[:200]}...")

    # Fetch and collect metrics
    metrics = get_latest_trace_metrics(
        agent_name="customer-support-v6-gateway",
        wait_seconds=5,
        max_retries=5,
        timeout_seconds=120,
    )
    print_metrics(metrics, test_name)
    collect_metric(metrics, test_name)

V6 GATEWAY WITH SEMANTIC TOOL SEARCH
Each query uses x_amz_bedrock_agentcore_search to find relevant tools.
Watch two things:
  - tools_loaded     : semantic search picks the relevant tool(s)
  - skill_activated  : the device-troubleshooting skill should fire ONLY
                       on the malfunction query, and stay dormant otherwise

----------------------------------------------------------------------
Test: Return Policy
Query: What is your return policy for laptops?
----------------------------------------------------------------------
Tools Loaded:     1 (via semantic search)
Skill Activated:  False (device-troubleshooting)
Response: - **answer:** Here's our return policy for laptops:

  **Return Window:** 30 days from purchase

  **Condition Requirements:**
  - Original packaging and all accessories included
  - No physical damag...

                    LANGFUSE METRICS
  Test:          Return Policy
  Trace ID:      1aeb13d4d167f7b7e7e775b20a916d2b
-------------------------

Two things to read off the run above:

- **`Tools Loaded: 1`** — for every prompt, semantic search returns the single most relevant tool and the agent calls it. Progressive disclosure of tools.
- **`Skill Activated`** — `True` only for the Technical Support query, `False` everywhere else. The troubleshooting method loaded on demand and stayed quiet on policy, product, and greeting queries. That quiet is exactly what makes the Part A savings real.

In [22]:
# Print summary table
print_metrics_table()

# Save metrics for comparison in later notebooks
from utils.langfuse_metrics import save_metrics

save_metrics("v6")


                                  METRICS SUMMARY
               Test Latency    Cost Input Output Cache Read Tokens Cache Write Tokens
      Return Policy   6.89s $0.0058 4,149    332                 0                  0
       Product Info   8.79s $0.0098 1,216    380             1,486              1,486
  Technical Support   9.77s $0.0134 2,862    263             2,966              1,483
Multi-part Question   9.20s $0.0066 4,239    465                 0                  0
   General Question   4.64s $0.0040   468    173                 0              1,504
---------------------------------------------------------------------------------------------------------
  TOTALS: Latency(avg): 7.86s | Cost: $0.0396 | Input: 12,934 | Output: 1,613
          Cache Read Tokens: 4,452 | Cache Write Tokens: 4,473

Metrics saved as 'v6' → .lab_metrics.json


## Step 13: Compare with v4 (Routing)

Enter your metrics from Lab 04 (v4 routing) to compare with v6 gateway results.

In [23]:
from utils.langfuse_metrics import load_metrics, print_comparison

# Load metrics from Lab 04 (saved automatically when you ran print_metrics_table())
v4 = load_metrics("v4")

# Or enter manually if Lab 04 metrics weren't saved:
# v4 = {"total_cost": 0.0393, "avg_latency": 8.88, "total_input_tokens": 14008, "total_output_tokens": 1895}

# Print comparison (current metrics auto-calculated from collected)
print_comparison(
    prev_name="v4 (Routing)",
    curr_name="v6 (Gateway)",
    prev_cost=v4["total_cost"],
    prev_latency=v4["avg_latency"],
    prev_input_tokens=v4["total_input_tokens"],
    prev_output_tokens=v4["total_output_tokens"],
)

Loaded 'v4' metrics: cost=$0.0450, latency=6.92s, input=19,885, output=1,618
  V4 (ROUTING) vs V6 (GATEWAY) COMPARISON
Metric                     v4 (Routing)       v6 (Gateway)       Change
----------------------------------------------------------------------
Total Cost           $           0.0450 $           0.0396       -12.0%
Avg Latency (s)                    6.92               7.86       +13.6%
Input Tokens                     19,885             12,934       -35.0%
Output Tokens                     1,618              1,613        -0.3%

Result: 12.0% cost reduction, 13.6% latency increase


### Key Observation

v6 beats v4 by a clear margin above. But be careful about why, because it isn't the Gateway.

Almost all of that input-token drop comes from the Part A skill offload. v4 carried the full troubleshooting method in its prompt on every query. v6 moved it into a skill that only loads on malfunction queries, so every other query's prompt is about 1,136 tokens lighter. That's the bulk of the −43%, and it's a context refactor, not tool search. Caching pitches in too: with the core prompt cached, repeat queries re-read it at roughly a tenth of the cost.

Semantic tool search contributes basically nothing here, so don't read it into these numbers. At 4 tools the definitions are tiny either way, and the per-query swing above is mostly tool results (a long return-policy answer vs. a short greeting) plus whichever model the router picked. Part B is structural. It pays off at scale: 100 tools at ~500 tokens each is ~50K tokens of overhead per request (Anthropic measured 58 tools at ~55K). Semantic search keeps that flat as the catalog grows, and picks tools more accurately.

So the measurable win on this small agent is the skill. The Gateway is what you reach for once the tool count climbs.

> **Gateway pricing:** the comparison here is LLM token cost only. AgentCore Gateway bills its own way (per Search call, per InvokeTool, per tool indexed), small per call but worth factoring into a real cost estimate.

## Summary

One idea ran through this whole lab: keep the pointer in context, load the full thing on demand. It's the core move in harness engineering, and we applied it on both sides of the agent.

### Part A — Skills (instructions)

We pulled a long troubleshooting method out of the system prompt and put it in an on-demand `device-troubleshooting` skill.

- v1 inlined it, v2–v5 carried it in the shared prompt (paid every query), v6 dropped it to a skill that only loads when a malfunction query needs it.
- Every v6 query's prompt is about 1,136 tokens lighter (the skill adds back ~312, and only when it activates). It stayed quiet on every non-troubleshooting query, because its `description` names the symptoms it's for and the things it isn't.
- This is the saving you can actually measure on a small agent today.

### Part B — Semantic Tool Search (tools)

We moved the tools out of the agent process and into AgentCore Gateway, a shared MCP server.

1. Created an AgentCore Gateway with semantic search turned on.
2. Deployed a Lambda holding all four tools (`get_return_policy`, `get_product_info`, `get_technical_support`, `web_search`).
3. Registered tool schemas with descriptions written for semantic matching, backed by that Lambda.
4. Deployed a v6 agent that searches for the relevant tool per query with `x_amz_bedrock_agentcore_search` and loads only what it needs.

The Gateway earns its keep when you have a lot of tools (50–100+), several agents sharing them, separate teams owning them, or a need for JWT access control. At 4 tools the token win is small. At enterprise scale, holding that tool-definition overhead flat is the whole point.

### The takeaway

Skills and semantic tool search are the same move pointed at two different things, instructions and tools. Any time your context fills up with stuff the current query won't use, ask whether you can load it on demand instead of carrying it every turn.

---

**Next:** Lab 07 runs full evaluations across all the agent versions to measure what our optimizations added up to.

## Cleanup (Optional)

In [24]:
# # Uncomment to delete resources created in this lab
# Note: Cognito resources are managed by infrastructure stack
# agentcore_runtime.destroy(delete_ecr_repo=True)
# print(f"Deleted agent and ECR repository: {agent_name}")

# # Delete the Gateway and its targets
# from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient
# gw_client = GatewayClient(region_name=region)
# gw_client.delete_gateway(name=gateway_name, skip_resource_in_use=True)
# print(f"Deleted Gateway: {gateway_name}")

# # Delete the Lambda function and IAM role created in Step 5
# lambda_client.delete_function(FunctionName="workshop-gateway-tools")
# print("Deleted Lambda: workshop-gateway-tools")
# iam_client.detach_role_policy(
#     RoleName="workshop-gateway-lambda-role",
#     PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
# )
# iam_client.delete_role_policy(RoleName="workshop-gateway-lambda-role", PolicyName="GatewayToolsPolicy")
# iam_client.delete_role(RoleName="workshop-gateway-lambda-role")
# print("Deleted IAM role: workshop-gateway-lambda-role")

# # Remove the inline policy added to the Gateway role
# gateway_role_name = get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role").split("/")[-1]
# iam_client.delete_role_policy(RoleName=gateway_role_name, PolicyName="InvokeGatewayLambdaPolicy")
# print("Removed InvokeGatewayLambdaPolicy from Gateway role")